# Notebook 2: Build the Graph

We load the raw files into a Neo4j graph using the DuckDB procedure. Zone lookup data becomes `Zone` nodes; aggregated trip statistics become `YELLOW_TRIP` and `GREEN_TRIP` relationships between zones. All file reading is done inside `duckdb.query()` called from Cypher — Neo4j drives the graph writes while DuckDB handles the analytical aggregation directly inside the database process.

The graph model is:

```
(:Zone)-[:YELLOW_TRIP]->(:Zone)
(:Zone)-[:GREEN_TRIP]->(:Zone)
```

Zone properties: `locationId`, `zone`, `borough`, `serviceZone`  
Relationship properties: `count`, `avgDistance`, `avgFare`, `avgDurationMinutes`, `totalPassengers`

## Prerequisites

Set the following environment variables before starting Jupyter:

```shell
export NEO4J_URI="bolt://localhost:7687"
export NEO4J_USERNAME="neo4j"
export NEO4J_PASSWORD="your-password"
export NEO4J_DATABASE="neo4j"
```

Run Notebook 1 first to download the raw data files.

## 1. Install dependencies

In [1]:
%pip install neo4j==6.1.0 --quiet

Note: you may need to restart the kernel to use updated packages.


## 2. Imports and configuration

In [2]:
import os

from neo4j import GraphDatabase

In [3]:
NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]

RAW_DIR = os.path.expanduser("~/neo4j-duckdb-taxi/data/raw")

YELLOW_PARQUET = os.path.join(RAW_DIR, "yellow_tripdata_2024-01.parquet")
GREEN_PARQUET  = os.path.join(RAW_DIR, "green_tripdata_2024-01.parquet")
ZONE_CSV       = os.path.join(RAW_DIR, "taxi+_zone_lookup.csv")

print("Credentials set.")

Credentials set.


## 3. Connect to Neo4j

In [4]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

print(driver.verify_connectivity())  # None is expected
print("Connection created.")

None
Connection created.


## 4. Clear and prepare the database

In [5]:
def clear_database(session):
    # Drop all constraints
    constraints = session.run("SHOW CONSTRAINTS YIELD name RETURN name").data()
    for c in constraints:
        session.run(f"DROP CONSTRAINT {c['name']} IF EXISTS")

    # Drop all non-lookup indexes
    indexes = session.run("""
        SHOW INDEXES YIELD name, type
        WHERE type <> 'LOOKUP'
        RETURN name
    """).data()
    for idx in indexes:
        session.run(f"DROP INDEX {idx['name']} IF EXISTS")

    # Delete all nodes and relationships in batches
    while True:
        result = session.run("""
            MATCH (n) WITH n LIMIT 1000
            DETACH DELETE n
            RETURN count(*) AS deleted
        """)
        if result.single()["deleted"] == 0:
            break

    print("Database cleared -- constraints, indexes and data removed.")

with driver.session(database=NEO4J_DATABASE) as session:
    clear_database(session)

Database cleared -- constraints, indexes and data removed.


## 5. Create constraint and index

In [6]:
with driver.session(database=NEO4J_DATABASE) as session:
    session.run("""
        CREATE CONSTRAINT zone_location_id IF NOT EXISTS
        FOR (z:Zone) REQUIRE z.locationId IS UNIQUE
    """)
    session.run("""
        CREATE INDEX zone_borough IF NOT EXISTS
        FOR (z:Zone) ON (z.borough)
    """)
    print("Constraint and index created.")

Constraint and index created.


## 6. Load Zone nodes

In [7]:
# Load Zone nodes -- read the CSV via duckdb.query() called from Cypher.
# Paths use double quotes inside the single-quoted SQL string to avoid
# Cypher parsing issues with special characters in filenames.
with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run(f"""
        CALL duckdb.query(
            'SELECT LocationID, Zone, Borough, service_zone
             FROM read_csv_auto("{ZONE_CSV}")
             ORDER BY LocationID'
        )
        YIELD row
        RETURN row.LocationID   AS locationId,
               row.Zone         AS zone,
               row.Borough      AS borough,
               row.service_zone AS serviceZone
    """)
    zones = result.data()

with driver.session(database=NEO4J_DATABASE) as session:
    for row in zones:
        session.run("""
            MERGE (z:Zone {locationId: $locationId})
            SET z.zone        = $zone,
                z.borough     = $borough,
                z.serviceZone = $serviceZone
        """, locationId=int(row["locationId"]), zone=row["zone"],
             borough=row["borough"], serviceZone=row["serviceZone"])

print(f"Loaded {len(zones)} Zone nodes.")

Loaded 265 Zone nodes.


## 7. Aggregate yellow trips

In [8]:
# Aggregate yellow trips via duckdb.query() called from Cypher.
# Duration uses epoch_ms arithmetic to avoid nested string literals.
with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run(f"""
        CALL duckdb.query(
            'WITH sample AS (
                SELECT * FROM read_parquet("{YELLOW_PARQUET}")
                USING SAMPLE 100000 ROWS
            )
            SELECT
                PULocationID,
                DOLocationID,
                COUNT(*) AS trip_count,
                ROUND(AVG(trip_distance), 2) AS avg_distance,
                ROUND(AVG(fare_amount), 2) AS avg_fare,
                ROUND(AVG((epoch_ms(tpep_dropoff_datetime) - epoch_ms(tpep_pickup_datetime)) / 60000.0), 2) AS avg_duration_minutes,
                SUM(passenger_count) AS total_passengers
            FROM sample
            WHERE PULocationID IS NOT NULL
              AND DOLocationID IS NOT NULL
            GROUP BY PULocationID, DOLocationID'
        )
        YIELD row
        RETURN row.PULocationID         AS pu,
               row.DOLocationID         AS do,
               row.trip_count           AS tripCount,
               row.avg_distance         AS avgDistance,
               row.avg_fare             AS avgFare,
               row.avg_duration_minutes AS avgDurationMinutes,
               row.total_passengers     AS totalPassengers
    """)

    yellow_agg = [
        (r["pu"], r["do"], r["tripCount"], r["avgDistance"],
         r["avgFare"], r["avgDurationMinutes"], r["totalPassengers"])
        for r in result
    ]

print(f"Yellow aggregated zone pairs: {len(yellow_agg):,}")

Yellow aggregated zone pairs: 7,031


## 8. Aggregate green trips

In [9]:
# Aggregate green trips via duckdb.query() called from Cypher.
with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run(f"""
        CALL duckdb.query(
            'SELECT
                PULocationID,
                DOLocationID,
                COUNT(*) AS trip_count,
                ROUND(AVG(trip_distance), 2) AS avg_distance,
                ROUND(AVG(fare_amount), 2) AS avg_fare,
                ROUND(AVG((epoch_ms(lpep_dropoff_datetime) - epoch_ms(lpep_pickup_datetime)) / 60000.0), 2) AS avg_duration_minutes,
                SUM(passenger_count) AS total_passengers
            FROM read_parquet("{GREEN_PARQUET}")
            WHERE PULocationID IS NOT NULL
              AND DOLocationID IS NOT NULL
            GROUP BY PULocationID, DOLocationID'
        )
        YIELD row
        RETURN row.PULocationID         AS pu,
               row.DOLocationID         AS do,
               row.trip_count           AS tripCount,
               row.avg_distance         AS avgDistance,
               row.avg_fare             AS avgFare,
               row.avg_duration_minutes AS avgDurationMinutes,
               row.total_passengers     AS totalPassengers
    """)

    green_agg = [
        (r["pu"], r["do"], r["tripCount"], r["avgDistance"],
         r["avgFare"], r["avgDurationMinutes"], r["totalPassengers"])
        for r in result
    ]

print(f"Green aggregated zone pairs: {len(green_agg):,}")

Green aggregated zone pairs: 5,314


## 9. Load trip relationships

In [10]:
def load_trips(session, rows, rel_type, batch_size=500):
    total = 0
    for i in range(0, len(rows), batch_size):
        batch = [
            {
                "puId":               int(r[0]),
                "doId":               int(r[1]),
                "count":              int(r[2]),
                "avgDistance":        float(r[3]) if r[3] is not None else 0.0,
                "avgFare":            float(r[4]) if r[4] is not None else 0.0,
                "avgDurationMinutes": float(r[5]) if r[5] is not None else 0.0,
                "totalPassengers":    int(r[6])   if r[6] is not None else 0,
            }
            for r in rows[i:i + batch_size]
        ]
        session.run(f"""
            UNWIND $batch AS row
            MATCH (pu:Zone {{locationId: row.puId}})
            MATCH (do:Zone {{locationId: row.doId}})
            MERGE (pu)-[t:{rel_type}]->(do)
            SET t.count              = row.count,
                t.avgDistance        = row.avgDistance,
                t.avgFare            = row.avgFare,
                t.avgDurationMinutes = row.avgDurationMinutes,
                t.totalPassengers    = row.totalPassengers
        """, batch=batch)
        total += len(batch)
        print(f"  {rel_type}: {total:,} / {len(rows):,}", end="\r")
    print(f"  {rel_type}: {total:,} relationships loaded.")

with driver.session(database=NEO4J_DATABASE) as session:
    load_trips(session, yellow_agg, "YELLOW_TRIP")
    load_trips(session, green_agg,  "GREEN_TRIP")

  YELLOW_TRIP: 7,031 relationships loaded.
  GREEN_TRIP: 5,314 relationships loaded.


## 10. Verify the graph

In [11]:
with driver.session(database=NEO4J_DATABASE) as session:
    counts = session.run("""
        MATCH (z:Zone)               WITH count(z) AS zones
        MATCH ()-[y:YELLOW_TRIP]->() WITH zones, count(y) AS yellow
        MATCH ()-[g:GREEN_TRIP]->()  RETURN zones, yellow, count(g) AS green
    """).single()
    print(f"Zone nodes:          {counts['zones']:,}")
    print(f"YELLOW_TRIP rels:    {counts['yellow']:,}")
    print(f"GREEN_TRIP rels:     {counts['green']:,}")

Zone nodes:          265
YELLOW_TRIP rels:    7,031
GREEN_TRIP rels:     5,314


## 11. Teardown

In [12]:
driver.close()
print("Connection closed.")

Connection closed.
